# TinyCeNN-LM — 8-Expert Top-2 MoE-CeNN

This notebook extends the rigorous plain CeNN checkpoint with **8 SwiGLU experts and Top-2 routing inside the shared recurrent CeNN cell**. All experts are initialized from the trained plain-CeNN FFN, so the MoE begins from the same function before expert specialization. It keeps the Transformer fully removed and uses the same 65,536-token rigorous benchmark.

In [ ]:
import subprocess, sys, pathlib, importlib
subprocess.run(['nvidia-smi'], check=False)
REPO_DIR = pathlib.Path('/content/TinyCeNN-LM')
if REPO_DIR.exists(): subprocess.run(['git','-C',str(REPO_DIR),'pull','--ff-only'], check=True)
else: subprocess.run(['git','clone','https://github.com/vtavakkoli/TinyCeNN-LM.git',str(REPO_DIR)], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO_DIR)], check=True)
SRC_DIR = REPO_DIR/'src'
if str(SRC_DIR) not in sys.path: sys.path.insert(0,str(SRC_DIR))
importlib.invalidate_caches()
import tinycenn_lm
print('TinyCeNN-LM:', tinycenn_lm.__file__)

## Hugging Face login
Add a Hugging Face **write token** to Colab Secrets as `HF_TOKEN`.

In [ ]:
from huggingface_hub import login, HfApi, snapshot_download
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception: hf_token = None
login(token=hf_token, add_to_git_credential=False) if hf_token else login()
api = HfApi(); hf_user = api.whoami()['name']; print('Logged in as:', hf_user)
SOURCE_HF_REPO = f'{hf_user}/TinyCeNN-LM-Distilled-v2'
plain_dir = snapshot_download(repo_id=SOURCE_HF_REPO, repo_type='model')
print('Plain CeNN baseline:', plain_dir)

In [ ]:
# 8 experts, Top-2 active per token; same 7-step/255-token CeNN context.
MAX_TOKENS=30_000_000; CONTEXT_LENGTH=256; BATCH_SIZE=4; GRAD_ACCUM=4
LEARNING_RATE=2e-4; CENN_STEPS=7; DILATIONS='1,2,4,8,16,32,64'
NUM_EXPERTS=8; TOP_K=2; ROUTER_AUX_WEIGHT=0.01; ROUTER_Z_WEIGHT=0.001
EVAL_BATCHES=64; EVAL_BATCH_SIZE=4; EVAL_EVERY=250; SHUFFLE_BUFFER=4096
OUTPUT_DIR=str(REPO_DIR/'checkpoints/cenn-moe-top2')
print('Held-out tokens:', EVAL_BATCHES*EVAL_BATCH_SIZE*CONTEXT_LENGTH)

In [ ]:
import shutil
for p in (pathlib.Path(OUTPUT_DIR), pathlib.Path(OUTPUT_DIR+'-best')):
    if p.exists(): shutil.rmtree(p)
cmd=[sys.executable,str(REPO_DIR/'scripts/train_moe_distill.py'),'--warmstart-plain-dir',plain_dir,'--max-tokens',str(MAX_TOKENS),'--context-length',str(CONTEXT_LENGTH),'--batch-size',str(BATCH_SIZE),'--grad-accum',str(GRAD_ACCUM),'--learning-rate',str(LEARNING_RATE),'--steps',str(CENN_STEPS),'--dilations',DILATIONS,'--num-experts',str(NUM_EXPERTS),'--top-k',str(TOP_K),'--router-aux-weight',str(ROUTER_AUX_WEIGHT),'--router-z-weight',str(ROUTER_Z_WEIGHT),'--shuffle-buffer',str(SHUFFLE_BUFFER),'--eval-batches',str(EVAL_BATCHES),'--eval-batch-size',str(EVAL_BATCH_SIZE),'--eval-every',str(EVAL_EVERY),'--output-dir',OUTPUT_DIR]
print('Running:', ' '.join(cmd)); subprocess.run(cmd,cwd=str(REPO_DIR),check=True)

In [ ]:
import json
report=json.loads((pathlib.Path(OUTPUT_DIR)/'moe_distillation_report.json').read_text())
summary={k:v for k,v in [('status',report['status']),('plain_start_ce',report['run_start']['student_ce']),('best_moe_ce',report['best']['student_ce']),('teacher_ce',report['best']['teacher_ce']),('best_moe_ppl',report['best']['student_ppl']),('teacher_ppl',report['best']['teacher_ppl']),('gap_recovery_percent',100*report['teacher_gap_recovery_fraction']),('warmstart_ce_delta',report['warmstart']['ce_absolute_delta']),('expert_fraction',report['best']['expert_fraction']),('router_entropy',report['best']['router_entropy'])]}
print(json.dumps(summary,indent=2))

In [ ]:
# Publish best checkpoint to Hugging Face.
best_dir=pathlib.Path(OUTPUT_DIR+'-best'); publish_dir=best_dir if best_dir.exists() else pathlib.Path(OUTPUT_DIR)
HF_REPO_ID=f'{hf_user}/TinyCeNN-LM-MoE-Top2'
card=f"# TinyCeNN-LM MoE Top-2\n\nTransformer-free CeNN with 8 SwiGLU experts, Top-2 routing.\n\n- Best CE: {report['best']['student_ce']:.6f}\n- Teacher CE: {report['best']['teacher_ce']:.6f}\n- Gap recovery: {100*report['teacher_gap_recovery_fraction']:.2f}%\n- Benchmark SHA256: `{report['evaluation']['fingerprint_sha256']}`\n"
(publish_dir/'README.md').write_text(card,encoding='utf-8')
(publish_dir/'moe_distillation_report.json').write_text(json.dumps(report,indent=2),encoding='utf-8')
api.create_repo(HF_REPO_ID,repo_type='model',private=False,exist_ok=True)
api.upload_folder(repo_id=HF_REPO_ID,repo_type='model',folder_path=str(publish_dir),commit_message='Upload 8-expert Top-2 MoE-CeNN')
print(f'https://huggingface.co/{HF_REPO_ID}')

In [ ]:
# Re-download and run structural + generation tests.
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from tinycenn_lm import build_moe_cenn_student, MoECeNNReplacementLayer
downloaded=snapshot_download(repo_id=HF_REPO_ID,repo_type='model')
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dtype=torch.bfloat16 if device.type=='cuda' and torch.cuda.is_bf16_supported() else (torch.float16 if device.type=='cuda' else torch.float32)
tokenizer=AutoTokenizer.from_pretrained(downloaded); student=build_moe_cenn_student(downloaded,device=device,dtype=dtype).eval()
assert len([m for m in student.modules() if isinstance(m,MoECeNNReplacementLayer)])==1
assert not any('self_attn' in n for n,_ in student.named_modules())
print('Transformer-free 8-expert Top-2 structure: PASS')
teacher=AutoModelForCausalLM.from_pretrained('arnir0/Tiny-LLM',dtype=dtype if device.type=='cuda' else None,attn_implementation='sdpa').to(device).eval()
for prompt in ['The capital of Austria is','Artificial intelligence can help','A small language model','In the future, efficient AI']:
    x=tokenizer(prompt,return_tensors='pt').to(device); kw=dict(max_new_tokens=40,do_sample=False,use_cache=False,pad_token_id=tokenizer.eos_token_id)
    with torch.inference_mode(): t=teacher.generate(**x,**kw); s=student.generate(**x,**kw)
    print('\nPROMPT:',prompt); print('TEACHER:',tokenizer.decode(t[0],skip_special_tokens=True)); print('MOE-CENN:',tokenizer.decode(s[0],skip_special_tokens=True))